In [1]:
from base64 import encode
from pydoc import classname
from tkinter.font import Font
from turtle import bgcolor, color, left, width
from unicodedata import category
from dash import dash, dcc, html, Input, Output, State, callback_context, dash_table
from matplotlib import backend_tools, style
from matplotlib.font_manager import afmFontProperty
from matplotlib.pyplot import legend, margins
from numpy import pad, size
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import openpyxl
import datetime
from plotly.subplots import make_subplots
import plotly.express.colors

In [2]:
###### 여기부터 시작!
original_data = pd.read_excel('./original_data/ori_4_7월_소파구매.xlsx')
original_data.head(1)

,oaid,oaid_type,dq_client_id,Mall 구분,dq_client_nm,item_title,brand,비교브랜드,refix_prc,가격대,100고가 제품,150 고가 제품,200 고가 제품,time_str,월구분,date,Unnamed: 16
0,c6f123d55d5112f56869280140bd4cbc,PC,dq_1179,가구자사몰,디쟈트,크로이 천연면피 통가죽 4인/ 6인 소파,디쟈트,기타,2790000,250만~300만원,100만원 이상,150만원 이상,200만원 이상,2022-04-01 15:16:38,4월,2022-04-01,4월1주차


In [3]:
######
# (                )
###추후에 일주일 단위로 데이터 업데이트 불러와서 original_data와 합치기

In [4]:
### 베이스 데이터

df_base = original_data[['item_title','dq_client_nm', 'brand','refix_prc','date' ]]
df_base['week'] = original_data['time_str'].dt.week
df_base['day'] = original_data['time_str'].dt.day
df_base['month'] = original_data['time_str'].dt.month
df_base['year'] = original_data['time_str'].dt.year
df_base['month_week'] = original_data['time_str'].dt.dayofweek
df_base['date'] = pd.to_datetime(df_base[['year', 'month', 'day']])
df_base['brand'] = df_base['brand'].str.strip()
df_base.head(5)

df_base.to_csv('./data/df_base.csv')
df_base.head(1)


C:\Users\yeana shin\AppData\Local\Temp\ipykernel_13044\945137960.py:4: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated. Please use Series.dt.isocalendar().week instead.
  df_base['week'] = original_data['time_str'].dt.week
C:\Users\yeana shin\AppData\Local\Temp\ipykernel_13044\945137960.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_base['week'] = original_data['time_str'].dt.week
C:\Users\yeana shin\AppData\Local\Temp\ipykernel_13044\945137960.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returnin

,item_title,dq_client_nm,brand,refix_prc,date,week,day,month,year,month_week
0,크로이 천연면피 통가죽 4인/ 6인 소파,디쟈트,디쟈트,2790000,2022-04-01,13,1,4,2022,4


In [5]:
df_base = pd.read_csv('./data/df_base.csv')

In [6]:
####### 업종별 데이터 생성

In [7]:
######  업종별 베이스 데이터

## 날짜별로 묶기 +  month, week 표기 

date_sum_prc = pd.DataFrame(df_base.groupby('date')['refix_prc'].sum())
date_count = pd.DataFrame(df_base.groupby('date')['item_title'].count())
result_1 = pd.concat([date_sum_prc,date_count],axis=1).reset_index()

## result_1은 저장해두고 추후 업데이트시 결과물과 합쳐서 다시 저장되게끔(이전날짜 불러오기용)
result_1['date'] = pd.to_datetime(result_1['date'])
result_1['month'] = result_1['date'].dt.month
result_1['week'] = result_1['date'].dt.isocalendar().week
result_1.head(3)

,date,refix_prc,item_title,month,week
0,2022-04-01,812473492,1590,4,13
1,2022-04-02,884462531,1535,4,13
2,2022-04-03,1194335646,2121,4,13


In [8]:
##########  월별 데이터
### 올해 데이터만 가져옴 + month로 묶기

df_month_thisyear = result_1[result_1['date'].dt.year == datetime.datetime.now().year].groupby('month').sum().reset_index()
del df_month_thisyear['week']
df_month_thisyear['year'] = datetime.datetime.now().year

### 작년 data 만들어주기(12월 만들기)
aaa = list(range(1,13))
df_month = pd.DataFrame(aaa)
df_month[1] = 1000000000
df_month[2] = 2000
df_month[3] = 2021	
df_month.columns = [['month', 'sum_refix_prc', 'sum_count', 'year']]
df_month

#### 만들어진 53주치 데이터프레임에 올해 데이터넣기
df_month.to_csv('./data/df_month.csv', index=False)
df_month = pd.read_csv('./data/df_month.csv')
df_month_thisyear.to_csv('./data/df_month_thisyear.csv', index=False)
df_month_thisyear = pd.read_csv('./data/df_month_thisyear.csv')
df_month
for i in list(df_month_thisyear['month']):
    df_month.iloc[i-1] = df_month_thisyear[df_month_thisyear['month']==i]

df_month.to_csv('./data/df_month.csv', index=False)
df_month = pd.read_csv('./data/df_month.csv')
# df3_month

In [9]:
###### fig_market_month 미리보기

month = list(df_month['month'].astype('str'))
month_sum_prc=list(df_month['sum_refix_prc'])
month_date_count=list(df_month['sum_count'])

month_colors_1 = ['rgb(123,213,195)'] * 12
month_colors_2 = ['gray'] * 12

last_year_month = list(df_month[df_month['year']==datetime.datetime.now().year-1]['month'])

for i in last_year_month:
    month_colors_1[i-1] = 'rgba(123,213,195,0.3)'
    month_colors_2[i-1] = 'rgba(128,128,128,0.3)'
    

fig_1 = go.Figure(data = [go.Bar(x=month,y=month_sum_prc,name='매출총합',marker_color=month_colors_1,customdata=month_date_count,text=month_sum_prc,textposition='auto',
                        hovertemplate="매출: %{y} 원<br>건수: %{customdata:,d} 건", yaxis='y',offsetgroup=1),
                        go.Bar(x=month,y=month_date_count,name='매출건수',marker_color=month_colors_2,customdata=month_sum_prc,text=month_date_count,textposition='inside',
                            hovertemplate="매출: %{customdata:,d} 원<br>건수: %{y} 건",yaxis='y2',offsetgroup=2)]
                        )
fig_1.update_layout(
                margin=go.layout.Margin(t=0,b=0,r=0),                
                plot_bgcolor='rgba(243, 249, 252, 0.92)',
                xaxis=dict(title='월',titlefont_size=12,tickfont_size=8),
                yaxis= dict(title='매출 총합(단위 : 억원)',titlefont_size=13,tickfont_size=8,tickformat=',',tickvals=[5000000000,10000000000,15000000000,20000000000,25000000000,30000000000],ticktext=[50,100,150,200,250,300]),
                            #ticktext=[20,40,60,80,100]),
                yaxis2=dict(title='매출 건수', overlaying='y',side='right', titlefont_size=13,tickfont_size=8,tickformat=','),
                legend=dict(x=0,y=1.0,bgcolor='rgba(255, 255, 255, 0)',bordercolor='rgba(255, 255, 255, 0)',),
                barmode='group',
                bargap=0.25, # gap between bars of adjacent location coordinates.
                bargroupgap=0, # gap between bars of the same location coordinate.
                height=385
                )   



In [89]:
###### fig_market_month 미리보기

month = list(df_month['month'].astype('str'))
month_sum_prc=list(df_month['sum_refix_prc'])
month_date_count=list(df_month['sum_count'])

month_colors_1 = ['rgb(123,213,195)'] * 12
month_colors_2 = ['gray'] * 12

last_year_month = list(df_month[df_month['year']==datetime.datetime.now().year-1]['month'])

for i in last_year_month:
    month_colors_1[i-1] = 'rgba(123,213,195,0.3)'
    month_colors_2[i-1] = 'rgba(128,128,128,0.3)'
    

fig_1 = go.Figure(data = [go.Scatter(x=month, y=month_sum_prc, name='2021년 매출', text=date_count, marker_color='rgb(55, 83, 109)',fill='tozeroy', fillcolor='rgba(0,100,80,0.3)',mode= 'none',legendrank=2),
                          go.Bar(x=month,y=month_sum_prc,name='2022년 매출',marker_color=month_colors_1,customdata=month_date_count,hovertemplate="매출: %{y} 원<br>건수: %{customdata:,d} 건",legendrank=1,yaxis='y2')
                        ])


# TODO: tickvals, ticktext 리스트 작성
# TODO: 예측치 모델 작성
# TODO: 현재 월 따로 표시

fig_1.update_layout(
                margin=go.layout.Margin(t=0,b=0,r=0),                
                plot_bgcolor='rgba(243, 249, 252, 0.92)',
                # xaxis=dict(title='월',titlefont_size=12,tickfont_size=8),
                yaxis= dict(tickfont_size=8,tickformat=',',tickvals=[5000000000,10000000000,15000000000,20000000000,25000000000,30000000000],ticktext=[50,100,150,200,250,300]),
                yaxis2=dict(tickfont_size=8,tickformat=',',tickvals=[5000000000,10000000000,15000000000,20000000000,25000000000,30000000000],ticktext=[50,100,150,200,250,300],overlaying='y'),
                # yaxis2=dict(title='매출 건수', overlaying='y',side='right', titlefont_size=13,tickfont_size=8,tickformat=','),
                legend=dict(x=0.4,y=-0.1,bgcolor='rgba(255, 255, 255, 0)',bordercolor='rgba(255, 255, 255, 0)',orientation="h"), # 가로 방향으로),
                barmode='group',
                bargap=0.5, # gap between bars of adjacent location coordinates.
                # bargroupgap=0, # gap between bars of the same location coordinate.
                height=385
                )   

In [39]:
fig = go.Figure((go.Scatter(x=month, y=month_sum_prc, name='매출건수', text=date_count, marker_color='rgb(55, 83, 109)',fill='tozeroy', fillcolor='rgba(0,100,80,0.2)')))
fig.show()

In [11]:
### 올해 데이터만 가져옴 + week로 묶기
df_week_thisyear = result_1[result_1['date'].dt.year == datetime.datetime.now().year].groupby('week').sum().reset_index()
del df_week_thisyear['month']
df_week_thisyear['year'] = datetime.datetime.now().year
# week_thisyear.head()

### 작년 data 만들어주기(53주차 만들기)
aaa = list(range(54))
df_week = pd.DataFrame(aaa)
df_week[1] = 500000000	
df_week[2] = 800	
df_week[3] = 2021	
df_week.columns = [['week', 'sum_refix_prc', 'sum_count', 'year']]
# df3_week.head()

#### 만들어진 53주치 데이터프레임에 올해 데이터넣기
df_week.to_csv('./data/df_week.csv', index=False)
df_week = pd.read_csv('./data/df_week.csv')
df_week_thisyear.to_csv('./data/df_week_thisyear.csv', index=False)
df_week_thisyear = pd.read_csv('./data/df_week_thisyear.csv')

for i in list(df_week_thisyear['week']):
    df_week.iloc[i] = df_week_thisyear[df_week_thisyear['week']==i]

df_week.to_csv('./data/df_week.csv', index=False)
df_week = pd.read_csv('./data/df_week.csv')
df_week

,week,sum_refix_prc,sum_count,year
0,0,500000000,800,2021
1,1,500000000,800,2021
2,2,500000000,800,2021
3,3,500000000,800,2021
4,4,500000000,800,2021
5,5,500000000,800,2021
6,6,500000000,800,2021
7,7,500000000,800,2021
8,8,500000000,800,2021
9,9,500000000,800,2021


In [12]:
###### fig_market_week 미리보기

week = list(df_week['week'].astype('str'))
week_sum_prc=list(df_week['sum_refix_prc'])
week_date_count=list(df_week['sum_count'])


week_colors_1 = ['rgb(115,198,217)'] * 54
week_colors_2 = ['gray'] * 54
last_year_week = list(df_week[df_week['year']==datetime.datetime.now().year-1]['week'])
for i in last_year_week:
    # colors_1[i] = 'lightslategray'
    week_colors_1[i] = 'rgba(115,198,217,0.3)'
    week_colors_2[i] = 'rgba(128,128,128,0.3)'
    opacity=0.6
    
fig_1 = go.Figure(data = [go.Bar(x=week,y=week_sum_prc,name='매출총합',marker_color=week_colors_1,customdata=week_date_count,
                        hovertemplate="매출: %{y} 원<br>건수: %{customdata:,d} 건", yaxis='y',offsetgroup=1),
                        go.Bar(x=week,y=week_date_count,name='매출건수',marker_color=week_colors_2,customdata=week_sum_prc,
                            hovertemplate="매출: %{customdata:,d} 원<br>건수: %{y} 건",yaxis='y2',offsetgroup=2)]
                        )
fig_1.update_layout(
                margin=go.layout.Margin(t=0,b=0,r=0),                
                plot_bgcolor='rgba(243, 249, 252, 0.92)',
                xaxis=dict(title='월',titlefont_size=12,tickfont_size=8),
                yaxis= dict(title='매출 총합(단위 : 억원)',titlefont_size=13,tickfont_size=8,tickformat=',',tickvals=[5000000000,10000000000,15000000000,20000000000,25000000000,30000000000],ticktext=[50,100,150,200,250,300]),
                yaxis2=dict(title='매출 건수', overlaying='y',side='right', titlefont_size=13,tickfont_size=8,tickformat=','),
                legend=dict(x=0,y=1.0,bgcolor='rgba(255, 255, 255, 0)',bordercolor='rgba(255, 255, 255, 0)',),
                barmode='group',
                bargap=0.25, # gap between bars of adjacent location coordinates.
                bargroupgap=0, # gap between bars of the same location coordinate.
                height=385
                )    


#########